In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
df = pd.read_csv('/kaggle/input/student-performance-dataset/ultimate_student_productivity_dataset_5000.csv')

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
cat_cols = df.describe(include='object').columns

In [ ]:
for col in cat_cols:
    print(f'{col} -- {df[col].unique()}')

In [ ]:
gender_map = {'Male':1, 'Female':0, 'Other':2}
acad_map = {'High School':0, 'Undergraduate':1, 'Postgraduate':2}
int_map = {'Good':1,  'Poor':0, 'Average':0.5}

In [ ]:
df['gender'] = df['gender'].map(gender_map)
df['academic_level'] = df['academic_level'].map(acad_map)
df['internet_quality'] = df['internet_quality'].map(int_map)

In [ ]:
df.head()

In [ ]:
df.corr()['exam_score'].sort_values(ascending=False).head(10)

## **Assumption-1:** Correlation with Target
we can see a good correlation between `target column` and `features`, **assumption 1 passed!**

In [ ]:
plt.figure(figsize=(16, 8))
correlation_matrix = df.corr()   
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', )

## **Assumption 2:** Multicollinearity
Observe the multiple redundancy we are facing here.
`study hours` is finely correlated with both `productivity_score` and `exam_score`, where `productivity_score` is again highly correlated with `exam_score`.
assumption 2 failed. But now?

In [ ]:
# Separate features and target
X = df.drop(columns=['exam_score', 'student_id'])  
y = df['exam_score']                 

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=0.95)  
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca)

In [ ]:
# explained variance
explained_variance = pca.explained_variance_ratio_
print("Explained variance by component:", explained_variance)

In [ ]:
columns = [f'feature_{i}' for i in range(1,17)]

In [ ]:
df_pca.columns = columns

In [ ]:
df_pca.head()

In [ ]:
df_pca['exam_score'] = y.reset_index(drop=True)

In [ ]:
plt.figure(figsize=(16, 8))
correlation_matrix =df_pca.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', )

### **Assumption 2:** succeeded!
Now we passed **the second assumption!**

In [ ]:

# Features and target
X = df_pca.drop(columns=['exam_score'])
y = df_pca['exam_score']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predictions
y_pred = lr.predict(X_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Coefficients:", lr.coef_)
print("Intercept:", lr.intercept_)
print("MSE:", mse)
print("R²:", r2)


Further improvements can be done by experimentation of encoding methods and feature engineering.

In [ ]:
residuals = y_test - y_pred
plt.scatter(y_pred, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Values")
plt.show()

## **Assumption 3:** Homoscedasticity
Slight failure is visible. A bias is present in a range of -10 to 10 predicted values showing either lack of features or a complete case of overfitting the behaviour of students scoring less.

**Solution**: Add more data for that range including a good feature engineering.

In [ ]:
X_const = sm.add_constant(X) 
model = sm.OLS(y, X_const).fit()
dw_stat = durbin_watson(model.resid)
print("Durbin-Watson statistic:", dw_stat)

## **Assumption 4:** Autocorrelation
tested and succeeded!

In [ ]:
# Compute residuals
residuals = y_test - y_pred

# Plot KDE
sns.kdeplot(residuals, fill=True)
plt.title("KDE of Residuals")
plt.xlabel("Residual")
plt.show()


## Assumption 5: Residual Normality
Very well done, **assumption 5 passeed!**